In [1]:
import re 

In [11]:
class RegexRouter:
    def __init__(self):
        self.patterns = [
            # Ticket-Year  dashboard
            {
                "name": "dashboard",
                "regex": re.compile(r'\b([A-Z]{1,5})\s+(\d{4})\b'),
                "format": "ROUTE:DASHBOARD|{0}|{1}"
            },
            {
                "name": "exact_metric",
                "regex": re.compile(r'\b([A-Z]{1,5})\s+(?:(?:in|for)?\s*(\d{4})\s+)?([Rr]evenue|[Nn]et\s+[Ii]ncome|[Gg]ross\s+[Mm]argin|[Oo]perating\s+[Mm]argin)(?:\s+(?:in|for)?\s*(\d{4}))?\b'),
                "format": "ROUTE:EXACT_METRIC|{0}|{2}|{1}{3}" # Handles year before or after metric
            },
            {
                "name": "sec_doc",
                "regex": re.compile(r'\b([A-Z]{1,5})\s+(10-K|10-Q|8-K)(?:\s+(\d{4}))?\b', re.IGNORECASE),
                "format": "ROUTE:FETCH_DOC|{0}|{1}|{2}"
            },
            {
                "name": "cik_lookup",
                "regex": re.compile(r'\bCIK\s+(\d{10})\b', re.IGNORECASE),
                "format": "ROUTE:CIK_LOOKUP|{0}"
            }
        ]
        
    def get(self,query:str) -> str | None :
        """
        Evaluates the query against per-compiled regex patterns 
        """
        
        for pattern in self.patterns:
            match = pattern["regex"].search(query)
            if match:
                groups = match.groups()
                
                if pattern["name"] == "exact_metric":
                    ticker = groups[0].upper()
                    metric = groups[2].lower()
                    year = groups[1] if groups[1] else groups[3]
                    return f"ROUTE:EXACT_METRIC|{ticker}|{metric}|{year}"
                
                safe_groups = [g if g else "None" for g in groups]
                if pattern["name"] in ["dashboard","sec_doc"]:
                    safe_groups[0] = safe_groups[0].upper()
                    
                return pattern["format"].format(*safe_groups)
        
        return None
        

In [ ]:
if __name__ == "__main__":
    import time

    print("Initializing Regex Router...")
    router = RegexRouter()
    print("Router ready.\n")

    tests = [
        "Can you show me the AAPL 2023 dashboard?",
        "What was TSLA net income in 2022?",
        "Fetch the META 10-K",
        "Search for CIK 0000320193 please",
        "Why did Apple's revenue drop in Q3?" 
    ]

    for i, test_query in enumerate(tests, 1):
        print(f"Test {i}: '{test_query}'")
        
        # Using nanoseconds for higher precision since regex is incredibly fast
        start = time.perf_counter()
        result = router.get(test_query)
        latency = (time.perf_counter() - start) * 1000
        
        print(f"  -> Result:  {result}")
        print(f"  -> Latency: {latency:.4f} ms\n")

Initializing Regex Router...
Router ready.

Test 1: 'Can you show me the AAPL 2023 dashboard?'
  -> Result:  ROUTE:DASHBOARD|AAPL|2023
  -> Latency: 0.0299 ms

Test 2: 'What was TSLA net income in 2022?'
  -> Result:  ROUTE:EXACT_METRIC|TSLA|net income|2022
  -> Latency: 0.0203 ms

Test 3: 'Fetch the META 10-K'
  -> Result:  ROUTE:FETCH_DOC|META|10-K|None
  -> Latency: 0.0245 ms

Test 4: 'Search for CIK 0000320193 please'
  -> Result:  ROUTE:CIK_LOOKUP|0000320193
  -> Latency: 0.0238 ms

Test 5: 'Why did Apple's revenue drop in Q3?'
  -> Result:  None
  -> Latency: 0.0157 ms



In [13]:
class FallbackChain:
    def __init__(self):
        self.handlers = []
        
    def add_handler(self,handler) -> None:
        """
        Appends a routing handler to the chain 
        the order of addition dictates the exact execution order.
        """
        self.handlers.append(handler)
        
    def route(self,query : str) -> str | None:
        
        for handler in self.handlers:
            result = handler.get(query)
            if result is not None:
                return result
            
        return None

In [14]:
import faiss
import redis
import numpy as np
from sentence_transformers import SentenceTransformer


# Semantic cache class
class SemanticCache:
    def __init__(self,threshold: float = 0.88, redis_host : str = 'localhost',redis_port: int = 6379):
        self.threshold = threshold
        
        # Encoder 
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        dim = self.encoder.get_embedding_dimension()
        
        self.index = faiss.IndexIDMap(faiss.IndexFlatIP(dim))
        
        # K/V store (redis)
        self.redis_client = redis.Redis(host=redis_host,port=redis_port,decode_responses=True)
        self.next_id = 0
    
    def get(self,query:str) -> str | None:
        if self.index.ntotal == 0:
            return None
        
        clean_query = query.strip().lower()
        
        query_vector = self.encoder.encode([clean_query],normalize_embeddings=True)    


        # Search returns distances and integer ids 
        scores , indices = self.index.search(query_vector, k=1)
        
        best_score = scores[0][0]
        best_id = indices[0][0]
        
        print(f"  [DEBUG] Query: '{query}' | Top Score: {best_score:.4f}")
        
        if best_score >= self.threshold:
            return self.redis_client.get(f"cache:{best_id}")
        
        return None
    
    
    def set(self,query:str,answer:str)-> None:
        clean_query = query.strip().lower()
        query_vector = self.encoder.encode([clean_query],normalize_embeddings=True)
        
        # Writee to redis first if this fails , execution halt 
        self.redis_client.set(f"cache:{self.next_id}",answer)
        
        # Write to faiss second if this fails we have a harmless orphand key in redis that will be safely overwitten on the next successful operation 
        vector_id = np.array([self.next_id],dtype=np.int64)
        self.index.add_with_ids(query_vector,vector_id)
        
        self.next_id += 1
        

/opt/miniconda3/envs/ml_month3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
if __name__ == "__main__":
    import time

    print("Initializing L0 Routing Layer...")
    regex_router = RegexRouter()
    semantic_cache = SemanticCache(threshold=0.88, redis_port=6380)
    
    l0_chain = FallbackChain()
    l0_chain.add_handler(regex_router)
    l0_chain.add_handler(semantic_cache)
    print("L0 Chain ready.\n")

    # Explicitly seed the cache to prevent the zero-count short-circuit
    print("Seeding Semantic Cache...")
    semantic_cache.set(
        "What was Apple's total revenue in 2023?", 
        "Apple's total net sales for 2023 was $383.28 billion."
    )
    print("Cache seeded.\n")

    tests = [
        "Can you show me the AAPL 2023 dashboard?", 
        "What was Apple's total revenue in 2023?",  
        "What was Apple's gross margin in 2023?"    
    ]

    for i, test_query in enumerate(tests, 1):
        print(f"Test {i}: '{test_query}'")
        
        start = time.perf_counter()
        result = l0_chain.route(test_query)
        latency = (time.perf_counter() - start) * 1000
        
        print(f"  -> Result:  {result}")
        print(f"  -> Latency: {latency:.4f} ms\n")

Initializing L0 Routing Layer...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3592.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


L0 Chain ready.

Seeding Semantic Cache...
Cache seeded.

Test 1: 'Can you show me the AAPL 2023 dashboard?'
  -> Result:  ROUTE:DASHBOARD|AAPL|2023
  -> Latency: 0.0684 ms

Test 2: 'What was Apple's total revenue in 2023?'
  [DEBUG] Query: 'What was Apple's total revenue in 2023?' | Top Score: 1.0000
  -> Result:  Apple's total net sales for 2023 was $383.28 billion.
  -> Latency: 63.8789 ms

Test 3: 'What was Apple's gross margin in 2023?'
  [DEBUG] Query: 'What was Apple's gross margin in 2023?' | Top Score: 0.8335
  -> Result:  None
  -> Latency: 430.1463 ms



In [21]:
def generate_cost_comparison():
    # Constants
    COST_L0 = 0.000
    COST_L1 = 0.002
    COST_L2 = 0.020
    BASELINE_COST = 100 * COST_L2

    # Scenarios: (Name, L0_hits, L1_hits, L2_hits)
    # L1 Router Only: 77 LITE, 23 L2 (20 PRO + 3 failures)
    # Full Tiered: 50 L0 hits. The remaining 50 follow the 77/23 split (39 L1, 11 L2).
    scenarios = [
        ("Naive Baseline", 0, 0, 100),
        ("L1 Router Only", 0, 77, 23),
        ("L0 + L2 Only", 50, 0, 50),
        ("Full Tiered System", 50, 39, 11)
    ]

    print(f"{'Scenario':<22} | {'L0 Hits':<7} | {'L1 Hits':<7} | {'L2 Hits':<7} | {'Total Cost':<10} | {'Reduction'}")
    print("-" * 75)

    for name, l0, l1, l2 in scenarios:
        total_cost = (l0 * COST_L0) + (l1 * COST_L1) + (l2 * COST_L2)
        reduction = ((BASELINE_COST - total_cost) / BASELINE_COST) * 100
        
        print(f"{name:<22} | {l0:<7} | {l1:<7} | {l2:<7} | ${total_cost:<9.2f} | {reduction:.1f}%")

if __name__ == "__main__":
    generate_cost_comparison()

Scenario               | L0 Hits | L1 Hits | L2 Hits | Total Cost | Reduction
---------------------------------------------------------------------------
Naive Baseline         | 0       | 0       | 100     | $2.00      | 0.0%
L1 Router Only         | 0       | 77      | 23      | $0.61      | 69.3%
L0 + L2 Only           | 50      | 0       | 50      | $1.00      | 50.0%
Full Tiered System     | 50      | 39      | 11      | $0.30      | 85.1%
